In [ ]:
import os
import re
import subprocess
import librosa
import shutil
import librosa.display

import soundfile as sf
import noisereduce as nr
import matplotlib.pyplot as plt
import numpy as np

from glob import glob

# Data Extraction and Cleansing

## Getting Script and Audio Files

In [ ]:
def extract_aphasia_clips(cha_file, video_file, output_dir, patient_prefix="patient"):
    """
    Parses a .cha transcript file and extracts corresponding audio clips
    from a source video into 22050Hz mono WAV files.
    """
    # --- 1. SETUP ---
    os.makedirs(output_dir, exist_ok=True)
    
    # Using the AphasiaBank format: "0_5000"
    timestamp_pattern = re.compile(r"(\d+)_(\d+)")
    clip_count = 0
    
    print(f"🎧 Scanning {os.path.basename(cha_file)} and extracting 22050Hz audio...")

    # --- 2. PARSING & EXTRACTION ---
    with open(cha_file, "r", encoding="utf-8") as f:
        for line in f:
            # ONLY look at the Patient's lines
            if line.startswith("*PAR:"):
                match = timestamp_pattern.search(line)
                
                if match:
                    start_ms = int(match.group(1))
                    end_ms = int(match.group(2))
                    
                    # Convert to seconds
                    start_sec = start_ms / 1000.0
                    duration_sec = (end_ms - start_ms) / 1000.0
                    
                    # Clean the transcripts
                    clean_text = line.replace("*PAR:", "").split("")[0].strip()
                    
                    # Skip completely empty lines just in case
                    if not clean_text:
                        continue

                    # Uses the dynamic prefix (e.g., patient1_0000, patient2_0000)
                    base_filename = f"{patient_prefix}_{clip_count:04d}"
                    out_wav = os.path.join(output_dir, f"{base_filename}.wav")
                    out_txt = os.path.join(output_dir, f"{base_filename}.txt")
                    
                    # Save the transcript to a text file
                    with open(out_txt, "w", encoding="utf-8") as txt_file:
                        txt_file.write(clean_text)
                    
                    # Crop and convert the audio using FFmpeg
                    cmd = [
                        "ffmpeg", "-y", 
                        "-ss", str(start_sec), 
                        "-i", video_file, 
                        "-t", str(duration_sec),
                        "-ar", "22050",          # Resample directly to 22050 Hz
                        "-ac", "1",              # Convert to Mono audio
                        "-c:a", "pcm_s16le",     # Uncompressed 16-bit WAV format
                        "-vn",                   # Strip the video track completely
                        out_wav
                    ]
                    
                    # Run FFmpeg silently
                    subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
                    
                    clip_count += 1
                    
    print(f"✅ Extraction complete! Saved {clip_count} clips for {patient_prefix}.")

In [ ]:
# MSUC01b (control 1)
extract_aphasia_clips(
    cha_file="/rds/general/user/ak8224/home/emoji_project/data/Control/MSUC01b.cha",
    video_file="/rds/general/user/ak8224/home/emoji_project/data/Control/MSUC01b.mp4",
    output_dir="/rds/general/user/ak8224/home/emoji_project/data/Control/control1",
    patient_prefix="control1"
)

In [ ]:
# MSUC02b (control 2)
extract_aphasia_clips(
    cha_file="/rds/general/user/ak8224/home/emoji_project/data/Control/MSUC02b.cha",
    video_file="/rds/general/user/ak8224/home/emoji_project/data/Control/MSUC02b.mp4",
    output_dir="/rds/general/user/ak8224/home/emoji_project/data/Control/control2",
    patient_prefix="control2"
)

In [ ]:
# MSUC07b (control 3)
extract_aphasia_clips(
    cha_file="/rds/general/user/ak8224/home/emoji_project/data/Control/MSUC07b.cha",
    video_file="/rds/general/user/ak8224/home/emoji_project/data/Control/MSUC07b.mp4",
    output_dir="/rds/general/user/ak8224/home/emoji_project/data/Control/control3",
    patient_prefix="control3"
)

In [ ]:
# MSUC08b (control 4)
extract_aphasia_clips(
    cha_file="/rds/general/user/ak8224/home/emoji_project/data/Control/MSUC08b.cha",
    video_file="/rds/general/user/ak8224/home/emoji_project/data/Control/MSUC08b.mp4",
    output_dir="/rds/general/user/ak8224/home/emoji_project/data/Control/control4",
    patient_prefix="control4"
)

In [ ]:
# MSUC09b (control 5)
extract_aphasia_clips(
    cha_file="/rds/general/user/ak8224/home/emoji_project/data/Control/MSUC09b.cha",
    video_file="/rds/general/user/ak8224/home/emoji_project/data/Control/MSUC09b.mp4",
    output_dir="/rds/general/user/ak8224/home/emoji_project/data/Control/control5",
    patient_prefix="control5"
)

## Noise Check

In [ ]:
def visualize_stft_spectrogram(audio_path):
    print(f"🎵 Loading audio: {os.path.basename(audio_path)}")
    
    # Load audio with soundfile 
    y, sr = sf.read(audio_path)
    if y.ndim > 1:
        y = y.mean(axis=1) # Convert to mono if it happens to be stereo
        
    # Calculate the STFT
    print("🧮 Calculating Linear STFT...")
    # STFT converts the 1D waveform into a 2D matrix of Frequencies over Time
    stft_matrix = librosa.stft(y)
    stft_magnitude = np.abs(stft_matrix)
    
    # Convert amplitude to decibels (dB) for visualization
    stft_db = librosa.amplitude_to_db(stft_magnitude, ref=np.max)
    
    # Plotting visualization
    fig, ax = plt.subplots(figsize=(10, 6))
    
    img = librosa.display.specshow(
        stft_db, 
        x_axis='time', 
        y_axis='hz', 
        sr=sr, 
        ax=ax, 
        cmap='magma'
    )
    
    ax.set_title(f'Linear STFT: {os.path.basename(audio_path)}', fontsize=16, fontweight='bold')
    fig.colorbar(img, ax=ax, format="%+2.0f dB")
    
    plt.tight_layout()
    plt.show()

    return stft_db, sr

In [ ]:
audio_file = "/rds/general/user/ak8224/home/emoji_project/data/Control/control1/selected/control1_0027.wav"
stft, sr = visualize_stft_spectrogram(audio_file)

In [ ]:
audio_file = "/rds/general/user/ak8224/home/emoji_project/data/Control/control2/selected/control2_0104.wav"
stft, sr = visualize_stft_spectrogram(audio_file)

In [ ]:
audio_file = "/rds/general/user/ak8224/home/emoji_project/data/Control/control3/selected/control3_0050.wav"
stft, sr = visualize_stft_spectrogram(audio_file)

In [ ]:
audio_file = "/rds/general/user/ak8224/home/emoji_project/data/Control/control4/selected/control4_0147.wav"
stft, sr = visualize_stft_spectrogram(audio_file)

In [ ]:
audio_file = "/rds/general/user/ak8224/home/emoji_project/data/Control/control5/selected/control5_0011.wav"
stft, sr = visualize_stft_spectrogram(audio_file)

## Batch Noise Cleansing

In [ ]:
def batch_clean_folder(reference_audio_path, input_folder, output_folder):
    print(f"🔍 Extracting noise profile from: {reference_audio_path}...")
    
    # 1. Load the reference audio to get the exact mic hiss
    y_ref, sr = librosa.load(reference_audio_path, sr=22050)
    
    # Grab the Noise Profile (Seconds x to y)
    noise_clip = y_ref[int(1.2 * sr):int(1.7 * sr)]
    
    # 2. Ensure output folder exists
    os.makedirs(output_folder, exist_ok=True)
    
    # 3. Find all .wav files in the input folder
    audio_files = glob(os.path.join(input_folder, "*.wav"))
    print(f"📁 Found {len(audio_files)} files to clean. Starting batch process...\n")
    
    # 4. Loop through and clean every file
    for file_path in audio_files:
        filename = os.path.basename(file_path)
        out_path = os.path.join(output_folder, filename)
        
        y_noisy, _ = librosa.load(file_path, sr=sr)
        
        y_clean = nr.reduce_noise(y=y_noisy, sr=sr, y_noise=noise_clip, prop_decrease=1.0)
        
        sf.write(out_path, y_clean, sr)
        
    print("\n🎉 Batch cleaning complete!")

In [ ]:
# Bad reference timing: 2.9 - 3.5 seconds
REFERENCE_FILE = "/rds/general/user/ak8224/home/emoji_project/data/Control/control2/selected/control2_0104.wav"
INPUT_DIR = "/rds/general/user/ak8224/home/emoji_project/data/Control/control2/selected"
OUTPUT_DIR = "/rds/general/user/ak8224/home/emoji_project/data/Control/control2/filtered"

batch_clean_folder(REFERENCE_FILE, INPUT_DIR, OUTPUT_DIR)

In [ ]:
# Bad reference timing: 2.9 - 3.2 seconds
REFERENCE_FILE = "/rds/general/user/ak8224/home/emoji_project/data/Control/control3/selected/control3_0050.wav"
INPUT_DIR = "/rds/general/user/ak8224/home/emoji_project/data/Control/control3/selected"
OUTPUT_DIR = "/rds/general/user/ak8224/home/emoji_project/data/Control/control3/filtered"

batch_clean_folder(REFERENCE_FILE, INPUT_DIR, OUTPUT_DIR)

In [ ]:
# Bad reference timing: 5.5 - 6.2 seconds
REFERENCE_FILE = "/rds/general/user/ak8224/home/emoji_project/data/Control/control4/selected/control4_0147.wav"
INPUT_DIR = "/rds/general/user/ak8224/home/emoji_project/data/Control/control4/selected"
OUTPUT_DIR = "/rds/general/user/ak8224/home/emoji_project/data/Control/control4/filtered"

batch_clean_folder(REFERENCE_FILE, INPUT_DIR, OUTPUT_DIR)

# Calculating Metrics

## Speech-to-Pause Ratio

In [ ]:
def analyze_single_vad(audio_path, top_db=15):
    # 1. Audio Load with soundfile
    y, sr = sf.read(audio_path)
    
    if y.ndim > 1:
        y = y.mean(axis=1)
        
    total_duration = len(y) / sr
    
    # 2. Run the VAD with 15dB threshold
    speech_intervals = librosa.effects.split(y, top_db=top_db)
    speech_samples = sum([end - start for start, end in speech_intervals])
    
    speech_duration = speech_samples / sr
    pause_duration = total_duration - speech_duration
    
    # 3. Calculate metrics
    speech_ratio = speech_duration / total_duration if total_duration > 0 else 0
    pause_ratio = pause_duration / total_duration if total_duration > 0 else 0
    
    return {
        "filename": os.path.basename(audio_path),
        "total_sec": total_duration,
        "speech_sec": speech_duration,
        "pause_sec": pause_duration,
        "speech_ratio": float(f"{speech_ratio:.4f}"),
        "pause_ratio": float(f"{pause_ratio:.4f}")
    }

def batch_analyze_directory_vad(directory_path):
    audio_files = glob(os.path.join(directory_path, "*.wav"))
    
    if not audio_files:
        print(f"❌ No .wav files found in {directory_path}")
        return
        
    print(f"🔍 Analyzing {len(audio_files)} files in directory: {directory_path}\n")
    
    total_dataset_duration = 0.0
    total_dataset_speech = 0.0
    total_dataset_pause = 0.0
    
    # Process each file and print individual stats
    for file_path in audio_files:
        stats = analyze_single_vad(file_path, top_db=15)
        
        # Aggregate global metrics
        total_dataset_duration += stats["total_sec"]
        total_dataset_speech += stats["speech_sec"]
        total_dataset_pause += stats["pause_sec"]
    
    # Calculate Global Ratios
    global_speech_ratio = total_dataset_speech / total_dataset_duration
    global_pause_ratio = total_dataset_pause / total_dataset_duration
    
    print("\n" + "="*50)
    print("📊 GLOBAL DATASET VAD BASELINE")
    print("="*50)
    print(f"Total Audio Processed : {total_dataset_duration / 60:.2f} minutes")
    print(f"Global Speech Ratio   : {global_speech_ratio:.4f}")
    print(f"Global Pause Ratio    : {global_pause_ratio:.4f}")
    print("="*50)

In [ ]:
# MSUC01b (control 1)
TARGET_DIRECTORY = "/rds/general/user/ak8224/home/emoji_project/data/Control/control1/selected"
batch_analyze_directory_vad(TARGET_DIRECTORY)

In [ ]:
# MSUC02b (control 2)
TARGET_DIRECTORY = "/rds/general/user/ak8224/home/emoji_project/data/Control/control2/filtered"
batch_analyze_directory_vad(TARGET_DIRECTORY)

In [ ]:
# MSUC07b (control 3)
TARGET_DIRECTORY = "/rds/general/user/ak8224/home/emoji_project/data/Control/control3/filtered"
batch_analyze_directory_vad(TARGET_DIRECTORY)

In [ ]:
# MSUC08b (control 4)
TARGET_DIRECTORY = "/rds/general/user/ak8224/home/emoji_project/data/Control/control4/filtered"
batch_analyze_directory_vad(TARGET_DIRECTORY)

In [ ]:
# MSUC09b (control 5)
TARGET_DIRECTORY = "/rds/general/user/ak8224/home/emoji_project/data/Control/control5/selected"
batch_analyze_directory_vad(TARGET_DIRECTORY)

## Disfluencies ("uhm" "uh" "um") and Formulaic Fillers ("you know")

In [ ]:
def normalize_disfluency_metrics(audio_dir, script_dir, new_script_dir):
    # 1. Create the new directory for the curated scripts
    os.makedirs(new_script_dir, exist_ok=True)
    
    # Grab only the manually curated audio files
    audio_files = glob(os.path.join(audio_dir, "*.wav"))
    
    if not audio_files:
        print(f"❌ No .wav files found in {audio_dir}")
        return
        
    print(f"🔍 Found {len(audio_files)} curated audio clips. Starting analysis...\n")
    
    filler_pattern = re.compile(r'\b(um|uh|uhm)\b', re.IGNORECASE)
    phrase_pattern = re.compile(r'\b(I don\'t know|I mean|let\'s see|you_know)\b', re.IGNORECASE)
    
    total_fillers = 0
    total_you_knows = 0
    total_duration_sec = 0.0
    files_processed = 0
    
    for audio_path in audio_files:
        base_name = os.path.basename(audio_path).replace(".wav", ".txt")
        old_txt_path = os.path.join(script_dir, base_name)
        new_txt_path = os.path.join(new_script_dir, base_name)
        
        # 1. Check if the script exists in the old directory and move it
        if os.path.exists(old_txt_path):
            shutil.move(old_txt_path, new_txt_path)
            target_txt = new_txt_path
        elif os.path.exists(new_txt_path):
            target_txt = new_txt_path
        else:
            print(f"⚠️ Warning: No matching transcript found for {base_name}")
            continue
            
        # 2. Get the exact audio duration instantly using soundfile.info
        duration = sf.info(audio_path).duration
        total_duration_sec += duration
        
        # 3. Read the text and count the disfluencies
        with open(target_txt, 'r', encoding='utf-8') as f_txt:
            text = f_txt.read()
            
        total_fillers += len(filler_pattern.findall(text))
        total_you_knows += len(phrase_pattern.findall(text))
        files_processed += 1
        
    # 4. Calculate the Per-Minute Rates
    total_duration_min = total_duration_sec / 60.0
    
    fillers_per_min = total_fillers / total_duration_min if total_duration_min > 0 else 0.0
    you_knows_per_min = total_you_knows / total_duration_min if total_duration_min > 0 else 0.0
    
    print("="*50)
    print("📊 CURATED DATASET DISFLUENCY METRICS")
    print("="*50)
    print(f"Valid Files Processed                               : {files_processed}")
    print(f"Total Audio Duration                                : {total_duration_min:.4f} minutes")
    print(f"Total 'Um/Uh/Uhm'                                   : {total_fillers}")
    print(f"Total 'You know/I don't know/I mean/Let's see'      : {total_you_knows}")
    print("-" * 50)
    print(f"Fillers Rate                                        : {fillers_per_min:.4f} per minute")
    print(f"Formulaic Phrase Rate                               : {you_knows_per_min:.4f} per minute")
    print("="*50)

In [ ]:
# MSUC01b (control 1)
AUDIO_DIR = "/rds/general/user/ak8224/home/emoji_project/data/Control/control1/selected"
OLD_SCRIPT_DIR = "/rds/general/user/ak8224/home/emoji_project/data/Control/control1/scripts"
NEW_SCRIPT_DIR = "/rds/general/user/ak8224/home/emoji_project/data/Control/control1/filtered_scripts"

normalize_disfluency_metrics(AUDIO_DIR, OLD_SCRIPT_DIR, NEW_SCRIPT_DIR)

In [ ]:
# MSUC02b (control 2)
AUDIO_DIR = "/rds/general/user/ak8224/home/emoji_project/data/Control/control2/filtered"
OLD_SCRIPT_DIR = "/rds/general/user/ak8224/home/emoji_project/data/Control/control2/scripts"
NEW_SCRIPT_DIR = "/rds/general/user/ak8224/home/emoji_project/data/Control/control2/filtered_scripts"

normalize_disfluency_metrics(AUDIO_DIR, OLD_SCRIPT_DIR, NEW_SCRIPT_DIR)

In [ ]:
# MSUC07b (control 3)
AUDIO_DIR = "/rds/general/user/ak8224/home/emoji_project/data/Control/control3/filtered"
OLD_SCRIPT_DIR = "/rds/general/user/ak8224/home/emoji_project/data/Control/control3/scripts"
NEW_SCRIPT_DIR = "/rds/general/user/ak8224/home/emoji_project/data/Control/control3/filtered_scripts"

normalize_disfluency_metrics(AUDIO_DIR, OLD_SCRIPT_DIR, NEW_SCRIPT_DIR)

In [ ]:
# MSUC08b (control 4)
AUDIO_DIR = "/rds/general/user/ak8224/home/emoji_project/data/Control/control4/filtered"
OLD_SCRIPT_DIR = "/rds/general/user/ak8224/home/emoji_project/data/Control/control4/scripts"
NEW_SCRIPT_DIR = "/rds/general/user/ak8224/home/emoji_project/data/Control/control4/filtered_scripts"

normalize_disfluency_metrics(AUDIO_DIR, OLD_SCRIPT_DIR, NEW_SCRIPT_DIR)

In [ ]:
# MSUC09b (control 5)
AUDIO_DIR = "/rds/general/user/ak8224/home/emoji_project/data/Control/control5/selected"
OLD_SCRIPT_DIR = "/rds/general/user/ak8224/home/emoji_project/data/Control/control5/scripts"
NEW_SCRIPT_DIR = "/rds/general/user/ak8224/home/emoji_project/data/Control/control5/filtered_scripts"

normalize_disfluency_metrics(AUDIO_DIR, OLD_SCRIPT_DIR, NEW_SCRIPT_DIR)

## MFCC Mean and Variance

In [ ]:
def analyze_single_mfcc_split(audio_path, top_db=15):
    y, sr = sf.read(audio_path)
    if y.ndim > 1:
        y = y.mean(axis=1)
        
    # 1. Strip the silence! 
    speech_intervals = librosa.effects.split(y, top_db=top_db)
    
    if len(speech_intervals) == 0:
        return None 
        
    speech_y = np.concatenate([y[start:end] for start, end in speech_intervals])
    
    # 2. Extract MFCCs
    mfccs = librosa.feature.mfcc(y=speech_y, sr=sr, n_mfcc=13)
    
    # 3. Drop the 0th coefficient (energy/volume)
    mfccs = mfccs[1:, :]
    
    # 4. Calculate the Standard Deviation for EACH of the 12 coefficients
    coeff_stds = np.std(mfccs, axis=1)
    
    # 5. Splitting the Analysis
    lower_tract_std = np.mean(coeff_stds[0:4])
    upper_tract_std = np.mean(coeff_stds[4:])
    
    return {
        "filename": os.path.basename(audio_path),
        "mfcc_matrix": mfccs, 
        "all_stds": coeff_stds.tolist(),
        "lower_tract_std": float(f"{lower_tract_std:.4f}"),
        "upper_tract_std": float(f"{upper_tract_std:.4f}"),
        "overall_std": float(f"{np.mean(coeff_stds):.4f}")
    }

def batch_analyze_directory_mfcc(directory_path, top_db=15):
    audio_files = glob(os.path.join(directory_path, "*.wav"))
    
    if not audio_files:
        print(f"❌ No .wav files found in {directory_path}")
        return None
    
    global_mfcc_list = []
    
    for file_path in audio_files:
        stats = analyze_single_mfcc_split(file_path, top_db) 
        
        if stats is not None:
            global_mfcc_list.append(stats["mfcc_matrix"])
            
    if not global_mfcc_list:
        print("No active speech found in dataset.")
        return None
        
    global_mfcc_matrix = np.concatenate(global_mfcc_list, axis=1)
    
    # Calculate Global Stats for all 12 coefficients
    global_coeff_stds = np.std(global_mfcc_matrix, axis=1)
    global_std = np.mean(global_coeff_stds)
    
    global_coeff_means = np.mean(global_mfcc_matrix, axis=1)
    global_mean = np.mean(global_coeff_means)
    
    global_lower_tract_mean = np.mean(global_coeff_means[0:4])
    global_lower_tract_std = np.mean(global_coeff_stds[0:4])
    
    global_upper_tract_mean = np.mean(global_coeff_means[4:])
    global_upper_tract_std = np.mean(global_coeff_stds[4:])
    
    print("\n" + "="*50)
    print(f"📊 GLOBAL MFCC BASELINE: {os.path.basename(os.path.normpath(directory_path))}")
    print("="*50)
    print(f"Total Active Speech Frames : {global_mfcc_matrix.shape[1]}")
    print(f"Global MFCC Mean           : {global_mean:.4f}")
    print(f"Global Overall Std Dev     : {global_std:.4f}")
    print("-" * 50)
    print(f"Lower Tract Mean (1-4)     : {global_lower_tract_mean:.4f}")
    print(f"Lower Tract Std Dev (1-4)  : {global_lower_tract_std:.4f}")
    print(f"Upper Tract Mean (5-12)    : {global_upper_tract_mean:.4f}")
    print(f"Upper Tract Std Dev (5-12) : {global_upper_tract_std:.4f}")
    print("="*50)

    return {
        "global_matrix": global_mfcc_matrix,
    }

In [ ]:
# MSUC01b (control 1)
TARGET_DIRECTORY = "/rds/general/user/ak8224/home/emoji_project/data/Control/control1/selected"
global_mfcc_control = batch_analyze_directory_mfcc(TARGET_DIRECTORY)

In [ ]:
# MSUC02b (control 2)
TARGET_DIRECTORY = "/rds/general/user/ak8224/home/emoji_project/data/Control/control2/filtered"
global_mfcc_control = batch_analyze_directory_mfcc(TARGET_DIRECTORY)

In [ ]:
# MSUC07b (control 3)
TARGET_DIRECTORY = "/rds/general/user/ak8224/home/emoji_project/data/Control/control3/filtered"
global_mfcc_control = batch_analyze_directory_mfcc(TARGET_DIRECTORY)

In [ ]:
# MSUC08b (control 4)
TARGET_DIRECTORY = "/rds/general/user/ak8224/home/emoji_project/data/Control/control4/filtered"
global_mfcc_control = batch_analyze_directory_mfcc(TARGET_DIRECTORY)

In [ ]:
# MSUC09b (control 5)
TARGET_DIRECTORY = "/rds/general/user/ak8224/home/emoji_project/data/Control/control5/selected"
global_mfcc_control = batch_analyze_directory_mfcc(TARGET_DIRECTORY)

## Word per Minutes (WPM)

In [ ]:
def calculate_directory_wpm(txt_dir, total_audio_duration_minutes):
    txt_files = glob(os.path.join(txt_dir, "*.txt"))
    
    if not txt_files:
        print(f"❌ No .txt files found in {txt_dir}")
        return 0, 0
        
    print(f"🧮 Calculating Effective WPM across {len(txt_files)} curated transcripts...")
    
    fillers_to_ignore = {'um', 'uh', 'uhm'}
    total_effective_words = 0
    
    for file_path in txt_files:
        with open(file_path, 'r', encoding='utf-8') as f:
            text = f.read()
            
        # 1. Strip gesture/vocal tags (e.g., '&=ges' is deleted, leaving 'excited')
        text = re.sub(r'&=\S+', '', text)
        
        # 2. Delete abandoned phrases and repetitions
        text = re.sub(r'<[^>]+>\s*\[//?\]', '', text) 
        
        # 3. Clean up any remaining formatting brackets and pauses
        text = re.sub(r'\[.*?\]', '', text) 
        text = re.sub(r'[<>]', '', text) 
        text = re.sub(r'\(\.{1,3}\)', '', text)
        
        # 4. Strip prefixes like &- and &+ so '&-um' becomes 'um'
        text = re.sub(r'&[-+]', '', text)
        
        # 5. Remove punctuation and split into raw words
        clean_text = re.sub(r'[^\w\s]', '', text.lower())
        words = clean_text.split()
        
        # 6. Apply the final word-by-word logic
        effective_words = []
        for word in words:
            # Skip the classic fillers
            if word in fillers_to_ignore:
                continue
                
            # Skip consecutive immediate duplicates
            if effective_words and word == effective_words[-1]:
                continue
                
            effective_words.append(word)
            
        total_effective_words += len(effective_words)
        
    # Calculate the final global WPM
    wpm = total_effective_words / total_audio_duration_minutes if total_audio_duration_minutes > 0 else 0
    
    print("\n" + "="*50)
    print("📊 EFFECTIVE WORDS PER MINUTE (WPM) BASELINE")
    print("="*50)
    print(f"Total Curated Files   : {len(txt_files)}")
    print(f"Total Audio Duration  : {total_audio_duration_minutes:.4f} minutes")
    print(f"Total Effective Words : {total_effective_words}")
    print("-" * 50)
    print(f"Effective WPM         : {wpm:.4f} words/min")
    print("="*50)
    
    return wpm, total_effective_words

In [ ]:
# MSUC01b (control 1)
NEW_SCRIPT_DIR = "/rds/general/user/ak8224/home/emoji_project/data/Control/control1/filtered_scripts"

wpm, words = calculate_directory_wpm(NEW_SCRIPT_DIR, total_audio_duration_minutes=5.6977)

In [ ]:
# MSUC02b (control 2)
NEW_SCRIPT_DIR = "/rds/general/user/ak8224/home/emoji_project/data/Control/control2/filtered_scripts"

wpm, words = calculate_directory_wpm(NEW_SCRIPT_DIR, total_audio_duration_minutes=5.0908)

In [ ]:
# MSUC07b (control 3)
NEW_SCRIPT_DIR = "/rds/general/user/ak8224/home/emoji_project/data/Control/control3/filtered_scripts"

wpm, words = calculate_directory_wpm(NEW_SCRIPT_DIR, total_audio_duration_minutes=4.1608)

In [ ]:
# MSUC08b (control 4)
NEW_SCRIPT_DIR = "/rds/general/user/ak8224/home/emoji_project/data/Control/control4/filtered_scripts"

wpm, words = calculate_directory_wpm(NEW_SCRIPT_DIR, total_audio_duration_minutes=5.9499)

In [ ]:
# MSUC09b (control 5)
NEW_SCRIPT_DIR = "/rds/general/user/ak8224/home/emoji_project/data/Control/control5/filtered_scripts"

wpm, words = calculate_directory_wpm(NEW_SCRIPT_DIR, total_audio_duration_minutes=5.1720)